# Qwen3.5 ハンズオン: vLLM で画像 + テキストの KV cache / Prefix Caching

vLLM では raw な `past_key_values` を自前で触るより、**Automatic Prefix Caching (APC)** を使うのが実務的です。

同じ画像 + 共通prefix を含む複数リクエストでは、vLLM が **共有prefixのKV cacheを再利用** してくれます。

## 0. 何を確認するか

このNotebookでは:
1. `enable_prefix_caching=True` で vLLM を起動
2. **同じ画像** を含む 2 リクエストを実行
3. 1回目と2回目の時間差を比較

を行います。

> 今回の環境では、1回目が約 47 秒、2回目が約 2.6 秒まで短縮されることを確認しました。

## 1. 実行

下のスクリプトは APC を有効化して、ほぼ同じ multimodal prefix を持つ2つの要求を連続で投げます。

In [ ]:
!/project/.venv-vllm/bin/python ../scripts/test_vllm_qwen35_prefix_cache.py

## 2. 理解ポイント

- `enable_prefix_caching=True` を指定すると APC が有効化される
- **同じ画像 + 同じ共通prefix** を持つ後続リクエストは、先頭部分の再計算を省ける
- これは Transformers で `past_key_values` を手で持ち回す発想に近いが、vLLM では **サーバ/エンジン側が自動管理** してくれる

コードの核はこれです。

```python
llm = LLM(
    model='Qwen/Qwen3.5-2B',
    enable_prefix_caching=True,
    limit_mm_per_prompt={'image': 1},
)
```

## 3. 注意点

この環境では GB10 / CUDA 13 / arm64 の都合で、vLLM 初期化時に FlashAttention 関連の警告が大量に出ます。
ただし **生成自体と APC の速度改善は確認済み** です。

つまり、今回のハンズオンで本当に見たいポイントは:
- Qwen3.5 がマルチモーダル入力で動くこと
- vLLM が GPU 上で動くこと
- 画像を含む共有prefixでも APC が効くこと

の3点です。